In [ ]:
import requests
import pandas as pd
import psycopg2
from tqdm import tqdm


DB_CONFIG = {
    "host": "10.0.0.3",
    "port": 5435,
    "database": "medical_db",
    "user": "medical_user",
    "password": "medical_password"
}

# Конфигурация API
API_URL = "http://localhost:8000/api/search"
THRESHOLD = 0.0 


# conn = psycopg2.connect(**DB_CONFIG)
# cur = conn.cursor()
# cur.execute("SELECT id, reg_id, descr FROM medical_rounds ORDER BY id")
# rows = cur.fetchall()
# cur.close()
# conn.close()

# all_rounds = pd.DataFrame(rows, columns=["round_id", "reg_id", "descr"])
# print(f"Всего записей в БД: {len(all_rounds)}")
# all_rounds


In [5]:
all_rounds = pd.read_parquet("rounds_from_db.parquet")
all_rounds

,round_id,reg_id,descr
0,1,3524784781654924090,Обход совместно с: к.м.н. доцентом А.Д. Кулаги...
1,2,12761509128594994479,"Обход совместно с: д.м.н. профессором, главным..."
2,3,18049789057289630658,Обход совместно с: зам. директора НИИДОГиТ по ...
3,4,4706675999578874978,Обход совместно с: зам. директора НИИДОГиТ по ...
4,5,6785292361299432353,Обход совместно с: директором НИИ ДОГиТ им. Р....
...,...,...,...
95,96,16805697338299485840,Обход совместно с: и.о.директора НИИ ДОГиТ им....
96,97,17606524827177219845,"Обход совместно с: \n, Очная консультация в НИ..."
97,98,15793279458000897145,Обход совместно с: директором НИИ ДОГиТ им. Р....
98,99,15153335036161099865,Обход совместно с: зав. отделением ТКМ для дет...


In [6]:
# 2. Определяем тестовые запросы
queries = [
    "пациенты с анемией и мукозитом",
    "пациенты с лейкозом, принимавшие антибиотики",
    "пациенты получившие алло-ТГСК и протокол Flu+Bu",
    "пациенты с острым миелоидным лейкозом  и делецией короткого плеча 12-ой хромосомы",
    "пациенты с РТПХ кожи принимавшие флударабин и бусульфан"
]


## MY APPROACH

In [ ]:

scores_dict = {q: {} for q in queries}

for query in tqdm(queries, desc="Processing queries"):
    response = requests.post(
        API_URL,
        json={"query": query},
        params={"threshold": THRESHOLD}
    )
    response.raise_for_status()
    data = response.json()
    for res in data['results']:
        round_id = res['round_id']
        score = res['score']
        scores_dict[query][round_id] = score


result_df = all_rounds[['round_id', 'reg_id', 'descr']].copy()
for query in queries:
    col_name = f"score_{query[:30].replace(' ', '_')}"  # короткое имя
    result_df[col_name] = result_df['round_id'].apply(lambda rid: scores_dict[query].get(rid, 0.0))

result_df.to_parquet("queries3res.parquet")
print("Результаты сохранены в queries3res.parquet")
print(result_df.head())

In [ ]:
result_df = pd.read_parquet("queries5res.parquet")
result_df

## VECTOR SEARCH

In [ ]:
# rag_eval.py
import psycopg2
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
import requests
from typing import List, Tuple, Dict


EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
CHUNK_SIZE = 512         
CHUNK_OVERLAP = 50
TOP_K = 20               


def split_into_chunks(text: str, chunk_size: int, chunk_overlap: int):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_text(text)
    return chunks


def build_faiss_index(chunk_texts: List[str], model) -> Tuple[faiss.IndexFlatIP, np.ndarray]:
    print("Encoding chunks for FAISS...")
    texts_with_prefix = [f"passage: {t}" for t in chunk_texts]
    embeddings = model.encode(texts_with_prefix, show_progress_bar=True, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    return index, embeddings

def search_query(query: str, index, model, chunk_metadata: List[dict], top_k: int) -> Dict[int, float]:
    query_emb = model.encode(f"query: {query}", convert_to_numpy=True)
    query_emb = query_emb.reshape(1, -1)
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, top_k)
    round_scores = {}
    for dist, idx in zip(distances[0], indices[0]):
        if dist <= 0:
            continue
        round_id = chunk_metadata[idx]["round_id"]
        if round_id not in round_scores or dist > round_scores[round_id]:
            round_scores[round_id] = float(dist)
    return round_scores


def main():
    
    rounds = all_rounds
    print(f"Loaded {len(rounds)} rounds.")

    chunk_texts = []
    chunk_metadata = [] 
    for i, row in tqdm(rounds.iterrows(), desc="Chunking"):
        if not row['descr'] or not isinstance(row['descr'], str):
            continue
        chunks = split_into_chunks(row['descr'], CHUNK_SIZE, CHUNK_OVERLAP)
        for chunk in chunks:
            chunk_texts.append(chunk)
            chunk_metadata.append({"round_id": row['round_id']})

    print(f"Total chunks: {len(chunk_texts)}")

   
    print(f"Loading embedding model {EMBEDDING_MODEL}...")
    model = SentenceTransformer(EMBEDDING_MODEL)


    index, _ = build_faiss_index(chunk_texts, model)


   
    all_round_ids = set(m["round_id"] for m in chunk_metadata)
    query_scores = {q: {} for q in queries}

    for q in queries:
        print(f"\nProcessing query: {q}")
        scores = search_query(q, index, model, chunk_metadata, TOP_K)
        
        for rid in all_round_ids:
            if rid not in scores:
                scores[rid] = 0.0
        query_scores[q] = scores


    result_df = pd.DataFrame({"round_id": list(all_round_ids)})
    for q in queries:
        col_name = f"rag_score_{q[:30].replace(' ', '_')}"
        result_df[col_name] = result_df["round_id"].apply(lambda x: query_scores[q].get(x, 0.0))


    result_df.to_parquet("rag_scores_results.parquet")
    print("\nSaved scores to rag_scores_results.parquet")
    print(result_df.head())

if __name__ == "__main__":
    main()

f:\ochir\vkr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 100 rounds.


Chunking: 100it [00:00, 14280.91it/s]

Total chunks: 580
Loading embedding model intfloat/multilingual-e5-large...



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1589.64it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding chunks for FAISS...


Batches: 100%|██████████| 19/19 [00:03<00:00,  5.41it/s]


Processing query: пациенты с анемией и мукозитом

Processing query: пациенты с лейкозом, принимавшие антибиотики

Processing query: пациенты получившие алло-ТГСК и протокол Flu+Bu

Processing query: пациенты с острым миелоидным лейкозом  и делецией короткого плеча 12-ой хромосомы

Processing query: пациенты с РТПХ кожи принимавшие флударабин и бусульфан

Saved scores to rag_scores_results.parquet
   round_id  rag_score_пациенты_с_анемией_и_мукозитом  \
0         1                                  0.000000   
1         2                                  0.000000   
2         3                                  0.840615   
3         4                                  0.000000   
4         5                                  0.843988   

   rag_score_пациенты_с_лейкозом,_принимавш  \
0                                  0.000000   
1                                  0.000000   
2                                  0.000000   
3                                  0.000000   
4                    

In [9]:
rag = pd.read_parquet("rag_scores_results.parquet")
rag

,round_id,rag_score_пациенты_с_анемией_и_мукозитом,"rag_score_пациенты_с_лейкозом,_принимавш",rag_score_пациенты_получившие_алло-ТГСК_,rag_score_пациенты_с_острым_миелоидным_л,rag_score_пациенты_с_РТПХ_кожи_принимавш
0,1,0.000000,0.000000,0.000000,0.000000,0.000000
1,2,0.000000,0.000000,0.000000,0.000000,0.000000
2,3,0.840615,0.000000,0.000000,0.000000,0.000000
3,4,0.000000,0.000000,0.000000,0.000000,0.000000
4,5,0.843988,0.842373,0.000000,0.848951,0.000000
...,...,...,...,...,...,...
95,96,0.000000,0.841382,0.858544,0.849101,0.000000
96,97,0.840056,0.000000,0.000000,0.000000,0.846783
97,98,0.000000,0.842311,0.000000,0.856844,0.000000
98,99,0.000000,0.000000,0.000000,0.000000,0.000000


## BM-25

In [ ]:

import psycopg2
import pandas as pd
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
import string
import re
from typing import List, Tuple, Dict


try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')


CHUNK_SIZE = 512      
CHUNK_OVERLAP = 50
TOP_K = 20



def split_into_chunks(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    """Разбивает текст на чанки с перекрытием."""
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - chunk_overlap
    return chunks


def tokenize(text: str) -> List[str]:
    """
    Токенизация текста для BM25.
    Приводим к нижнему регистру, удаляем пунктуацию и стоп-слова.
    """
  
    text = text.lower()
    
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    
    words = text.split()
    
    stop_words = set(stopwords.words('russian'))
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return words

def build_bm25_index(chunk_texts: List[str]):
    """Создаёт BM25 индекс на основе токенизированных чанков."""
    print("Tokenizing chunks for BM25...")
    tokenized_chunks = [tokenize(chunk) for chunk in tqdm(chunk_texts, desc="Tokenizing")]
    print("Building BM25 index...")
    bm25 = BM25Okapi(tokenized_chunks)
    return bm25

def search_query(query: str, bm25, chunk_metadata: List[dict], top_k: int) -> Dict[int, float]:
    """Возвращает словарь {round_id: max_score} для данного запроса."""
    query_tokens = tokenize(query)
    if not query_tokens:
        return {}
    scores = bm25.get_scores(query_tokens)
   
    top_indices = np.argsort(scores)[::-1][:top_k]

    round_scores = {}
    for idx in top_indices:
        if scores[idx] <= 0:
            continue
        round_id = chunk_metadata[idx]["round_id"]
        score = float(scores[idx])
        if round_id not in round_scores or score > round_scores[round_id]:
            round_scores[round_id] = score
    return round_scores

def normalize_scores(round_scores: Dict[int, float], all_round_ids: set) -> Dict[int, float]:
    """Нормализует скоры в диапазон [0, 1] и заполняет отсутствующие нулями."""
    max_score = max(round_scores.values()) if round_scores else 1.0
    norm_scores = {}
    for rid in all_round_ids:
        raw = round_scores.get(rid, 0.0)
        norm_scores[rid] = raw / max_score if max_score > 0 else 0.0
    return norm_scores

def main():
   
    print("Loading rounds from DB...")
    rounds = all_rounds
    print(f"Loaded {len(rounds)} rounds.")


    chunk_texts = []
    chunk_metadata = [] 
    for i, row in tqdm(rounds.iterrows(), desc="Chunking"):
        if not row['descr'] or not isinstance(row['descr'], str):
            continue
        chunks = split_into_chunks(row['descr'], CHUNK_SIZE, CHUNK_OVERLAP)
        for chunk in chunks:
            chunk_texts.append(chunk)
            chunk_metadata.append({"round_id": row['round_id']})

    print(f"Total chunks: {len(chunk_texts)}")

   
    bm25 = build_bm25_index(chunk_texts)


   
    all_round_ids = set(m["round_id"] for m in chunk_metadata)

  
    query_scores = {q: {} for q in queries}
    for q in queries:
        print(f"\nProcessing query: {q}")
        raw_scores = search_query(q, bm25, chunk_metadata, TOP_K)
        norm_scores = normalize_scores(raw_scores, all_round_ids)
        query_scores[q] = norm_scores

    
    result_df = pd.DataFrame({"round_id": list(all_round_ids)})
    for q in queries:
        col_name = f"bm25_score_{q[:30].replace(' ', '_')}"
        result_df[col_name] = result_df["round_id"].apply(lambda x: query_scores[q].get(x, 0.0))

    
    result_df.to_parquet("bm25_scores_results.parquet")
    print("\nScores saved to bm25_scores_results.parquet")
    print(result_df.head())

if __name__ == "__main__":
    main()

Loading rounds from DB...
Loaded 100 rounds.


Chunking: 100it [00:00, 32773.12it/s]


Total chunks: 529
Tokenizing chunks for BM25...


Tokenizing: 100%|██████████| 529/529 [00:00<00:00, 7243.98it/s]

Building BM25 index...

Processing query: пациенты с анемией и мукозитом

Processing query: пациенты с лейкозом, принимавшие антибиотики

Processing query: пациенты получившие алло-ТГСК и протокол Flu+Bu

Processing query: пациенты с острым миелоидным лейкозом  и делецией короткого плеча 12-ой хромосомы

Processing query: пациенты с РТПХ кожи принимавшие флударабин и бусульфан

Scores saved to bm25_scores_results.parquet
   round_id  bm25_score_пациенты_с_анемией_и_мукозитом  \
0         1                                        0.0   
1         2                                        0.0   
2         3                                        0.0   
3         4                                        0.0   
4         5                                        0.0   

   bm25_score_пациенты_с_лейкозом,_принимавш  \
0                                        0.0   
1                                        0.0   
2                                        0.0   
3                                 

## LLM

СКОЛЬКО В СРЕДНЕМ ЗАПИСЬ ОБХОДА

In [ ]:
import json
import requests
import time
import re
import numpy as np

MODEL_NAMES = ["qwen2.5:14b", "atla/selene-mini:latest", "gemma3:12b"]
PRIMARY_MODEL = "atla/selene-mini:latest"
ARBITER_MODELS = ["qwen2.5:14b", "gemma3:12b"]

def call_llm(system_prompt: str, user_prompt: str, model_name: str,
             temperature=0.0, is_json=False):
    payload = {
        "model": model_name,
        "prompt": user_prompt,
        "system": system_prompt,
        "stream": False,
        "keep_alive": 0,
        "options": {
            "temperature": temperature,
            "seed": 42,
            "num_ctx": 6144,
            "repetition_penalty": 1.05,
            "num_predict": 1024,
        },
    }
    if is_json:
        payload["format"] = "json"
    r = requests.post("http://localhost:11434/api/generate", json=payload, timeout=120)
    return r.json()["response"]

def get_score_llm(text: str, query: str, model_name: str) -> dict:
    """
    Возвращает словарь с ключами:
    - total_score: float 0..1 (нормированная релевантность)
    - concepts: список распознанных понятий и их оценок
    - raw_total: сырая сумма баллов
    - num_concepts: количество понятий в запросе (по версии модели)
    При ошибке возвращает None.
    """
    system_prompt = """Ты — эксперт по медицинской семантике (гематология, трансплантация костного мозга).
Оцени релевантность текста врачебного обхода поисковому запросу.

ВАЖНО: Извлекай понятия ТОЛЬКО из текста запроса, игнорируя общие слова (пациенты, с, и, или, в, на и т.п.).
Не добавляй понятия, которых нет в запросе. Сначала составь список понятий запроса, затем для каждого ищи совпадения в тексте.

Алгоритм:
1. Извлеки из запроса самостоятельные медицинские понятия.
   Пример: "пациенты с цитопенией" → ["цитопения"]
   Пример: "анемия и мукозит" → ["анемия", "мукозит"]

2. Для каждого понятия определи наличие в тексте по иерархии:
   - Точное совпадение, синоним, клинически эквивалентное описание → 5 баллов.
   - Гипоним (более узкое понятие) – если запрос является гиперонимом, то найденный гипоним даёт 5 баллов.
     Пример: запрос "цитопения" – текст "анемия" → 5.
   - Гипероним (более широкое понятие) – если запрос является гипонимом, то гипероним даёт 2 балла.
     Пример: запрос "анемия" – текст "цитопения" → 2.
   - Другие родственные, но не иерархически связанные понятия → 0 баллов.

3. Игнорируй отрицания, модальность. Упоминание термина считается присутствием.

4. Итоговая оценка (total_score) = сумма баллов по всем понятиям запроса.
   Максимально возможный балл = 5 × (количество найденных в запросе понятий).
   Нормируй итог, разделив total_score на максимальный балл.

Верни JSON в формате:
{
  "query_concepts": ["понятие1", "понятие2", ...],
  "concepts": [
    {"query_concept": "понятие", "matched_text": "найденный текст", "score": число}
  ],
  "total_score": число
}

Примеры:
Запрос: "пациенты с цитопенией"
Текст: "У пациента анемия"
query_concepts: ["цитопения"]
concepts: [{"query_concept": "цитопения", "matched_text": "анемия", "score": 5}]
total_score: 5

Запрос: "анемия, мукозит"
Текст: "Цитопения, стоматит"
query_concepts: ["анемия", "мукозит"]
concepts: [
    {"query_concept": "анемия", "matched_text": "цитопения", "score": 2},
    {"query_concept": "мукозит", "matched_text": "стоматит", "score": 5}
]
total_score: 7
"""

    user_prompt = f"Запрос: {query}\nТекст: {text}\nОцени релевантность. JSON:"

    raw = call_llm(system_prompt, user_prompt, model_name, is_json=True)
    try:
        data = json.loads(raw)
        concepts = data.get('concepts', [])
        total_raw = int(data.get('total_score', 0))
       
        num_concepts = max(len(concepts), 1)
        max_score = 5 * num_concepts
        norm_score = min(max(total_raw / max_score, 0.0), 1.0)
        return {
            'total_score': norm_score,
            'concepts': concepts,
            'raw_total': total_raw,
            'num_concepts': num_concepts
        }
    except Exception as e:
     
        nums = re.findall(r'\b([0-9]|10)\b', raw)
        if nums:
            total_raw = int(nums[-1])
            
            norm_score = min(total_raw / 10.0, 1.0)
            return {
                'total_score': norm_score,
                'concepts': [],
                'raw_total': total_raw,
                'num_concepts': 2
            }
        return None



def ensemble_score_weighted(text, query):
    primary = get_score_llm(text, query, PRIMARY_MODEL)
    if primary is None:
        primary_score = 0.0
    else:
        primary_score = primary['total_score']
    arb_scores = []
    for model in ARBITER_MODELS:
        res = get_score_llm(text, query, model)
        if res is not None:
            arb_scores.append(res['total_score'])
    if arb_scores:
       
        avg_arb = np.mean(arb_scores)
        return 0.6 * primary_score + 0.4 * avg_arb
    return primary_score



In [26]:
ensemble_score_weighted("у пациента обнаружена цитопения, после терапии, развитие мукозита ЖКТ", "пациенты с анемией")

np.float64(0.6799999999999999)

In [ ]:
res = get_score_llm("у пациента обнаружена анемия 2 ст, после терапии, развитие мукозита ЖКТ", 
                     "пациенты с цитопенией и лейкозом", 
                     PRIMARY_MODEL)
print(json.dumps(res, indent=2, ensure_ascii=False))

In [ ]:
llm_scores = all_rounds

llm_scores[queries[0][:30].replace(" ", "_")] = None
llm_scores[queries[1][:30].replace(" ", "_")] = None
llm_scores[queries[2][:30].replace(" ", "_")] = None


for j, query in enumerate(queries):
    for i, row in tqdm(llm_scores.iterrows()):
        llm_scores.at[i, query[:30].replace(" ", "_")] = ensemble_score_weighted(text=row['descr'], query=query)

llm_scores.to_parquet("llm_scores.parquet")

100it [2:04:58, 74.99s/it]
100it [2:05:33, 75.34s/it]
100it [2:10:38, 78.39s/it]


In [12]:
llm_scores = pd.read_parquet("llm_scores.parquet")
llm_scores.head()

llm_scores[queries[3][:30].replace(" ", "_")] = None
llm_scores[queries[4][:30].replace(" ", "_")] = None

for j, query in enumerate(queries):
    if j < 3:
        continue
    for i, row in tqdm(llm_scores.iterrows()):
        llm_scores.at[i, query[:30].replace(" ", "_")] = ensemble_score_weighted(text=row['descr'], query=query)

llm_scores.to_parquet("llm_scores.parquet")

0it [00:00, ?it/s]

100it [2:13:38, 80.19s/it]
100it [2:14:01, 80.42s/it]


In [13]:
llm_scores = pd.read_parquet("llm_scores.parquet")
llm_scores.head()

,round_id,reg_id,descr,пациенты_с_анемией_и_мукозитом,"пациенты_с_лейкозом,_принимавш",пациенты_получившие_алло-ТГСК_,пациенты_с_острым_миелоидным_л,пациенты_с_РТПХ_кожи_принимавш
0,1,3524784781654924090,Обход совместно с: к.м.н. доцентом А.Д. Кулаги...,0.00,0.34,0.0,0.04,0.00
1,2,12761509128594994479,"Обход совместно с: д.м.н. профессором, главным...",0.40,0.34,0.3,0.04,0.00
2,3,18049789057289630658,Обход совместно с: зам. директора НИИДОГиТ по ...,0.44,0.46,1.0,0.04,0.80
3,4,4706675999578874978,Обход совместно с: зам. директора НИИДОГиТ по ...,0.44,0.97,1.0,0.68,1.00
4,5,6785292361299432353,Обход совместно с: директором НИИ ДОГиТ им. Р....,1.00,1.00,1.0,0.34,0.68


## VAL

In [2]:
import pandas as pd

my = pd.read_parquet("queries3res.parquet")

my1 =  my.iloc[:, 3].to_list()
my2 =  my.iloc[:, 4].to_list()
my3 =  my.iloc[:, 5].to_list()

rag = pd.read_parquet("rag_scores_results.parquet")

rag1 = rag.iloc[:, 1].to_list()
rag2 = rag.iloc[:, 2].to_list()
rag3 = rag.iloc[:, 3].to_list()

bm = pd.read_parquet("bm25_scores_results.parquet")

bm1 = bm.iloc[:, 1].to_list()
bm2 = bm.iloc[:, 2].to_list()
bm3 = bm.iloc[:, 3].to_list()

llm_scores = pd.read_parquet("llm_scores.parquet")

llm1 = llm_scores.iloc[:, 3].to_list()
llm2 = llm_scores.iloc[:, 4].to_list()
llm3 = llm_scores.iloc[:, 5].to_list()

In [10]:
my

,round_id,reg_id,descr,score_пациенты_с_анемией_и_мукозитом,"score_пациенты_с_лейкозом,_принимавш",score_пациенты_получившие_алло-ТГСК_
0,1,3524784781654924090,Обход совместно с: к.м.н. доцентом А.Д. Кулаги...,0.000000,0.000000,0.0
1,2,12761509128594994479,"Обход совместно с: д.м.н. профессором, главным...",0.000000,0.000000,0.0
2,3,18049789057289630658,Обход совместно с: зам. директора НИИДОГиТ по ...,0.500000,0.427663,0.0
3,4,4706675999578874978,Обход совместно с: зам. директора НИИДОГиТ по ...,0.000000,0.434050,0.0
4,5,6785292361299432353,Обход совместно с: директором НИИ ДОГиТ им. Р....,0.915728,0.500000,0.0
...,...,...,...,...,...,...
95,96,16805697338299485840,Обход совместно с: и.о.директора НИИ ДОГиТ им....,0.500000,0.927663,0.5
96,97,17606524827177219845,"Обход совместно с: \n, Очная консультация в НИ...",0.500000,0.500000,0.5
97,98,15793279458000897145,Обход совместно с: директором НИИ ДОГиТ им. Р....,0.000000,0.500000,0.0
98,99,15153335036161099865,Обход совместно с: зав. отделением ТКМ для дет...,0.000000,0.431260,0.0


In [11]:
llm_scores

,round_id,reg_id,descr,пациенты_с_анемией_и_мукозитом,"пациенты_с_лейкозом,_принимавш",пациенты_получившие_алло-ТГСК_
0,1,3524784781654924090,Обход совместно с: к.м.н. доцентом А.Д. Кулаги...,0.00,0.34,0.00
1,2,12761509128594994479,"Обход совместно с: д.м.н. профессором, главным...",0.40,0.34,0.30
2,3,18049789057289630658,Обход совместно с: зам. директора НИИДОГиТ по ...,0.44,0.46,1.00
3,4,4706675999578874978,Обход совместно с: зам. директора НИИДОГиТ по ...,0.44,0.97,1.00
4,5,6785292361299432353,Обход совместно с: директором НИИ ДОГиТ им. Р....,1.00,1.00,1.00
...,...,...,...,...,...,...
95,96,16805697338299485840,Обход совместно с: и.о.директора НИИ ДОГиТ им....,0.94,1.00,0.94
96,97,17606524827177219845,"Обход совместно с: \n, Очная консультация в НИ...",0.70,1.00,1.00
97,98,15793279458000897145,Обход совместно с: директором НИИ ДОГиТ им. Р....,0.22,0.74,1.00
98,99,15153335036161099865,Обход совместно с: зав. отделением ТКМ для дет...,0.00,0.34,0.08


'Обход совместно с: д.м.н. профессором, главным детским онкологом СПб Ю.А. Пунановым руководителем отдела ДОГиТ д.м.н., профессором Л.С. Зубаровской,: \n, Пациентка 4х лет с диагнозом нейробластома правого надпочечника с поражением л/у забрюшинного пространства, средостения, таза, костного мозга. Состояние после комбинированного лечения (ПХТ по протоколу NB2004, оперативного лечения). По данным МЙБГ сцинтиграфии (28.04.2015), КТ грудной клетки, брюшной полости, малого таза с контрастированием у пациентки констатировано достижение полной ремиссии. Учитывая высокую группу риска пациентке по протоколу рекомендована интенсификация лечебной программы с выполнением ВДХТ с аутоТКМ. В настоящее время Д+1 после аутоТКМ, реинфузию и режим кондиционирования перенесла удовлетворительно. Начало периода цитопении.'

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score


my = pd.read_parquet("queries5res.parquet")
rag = pd.read_parquet("rag_scores_results.parquet")
bm = pd.read_parquet("bm25_scores_results.parquet")
llm_scores = pd.read_parquet("llm_scores.parquet")


my_scores = [my.iloc[:, 3].to_list(), my.iloc[:, 4].to_list(), my.iloc[:, 5].to_list(), my.iloc[:, 6].to_list(), my.iloc[:, 7].to_list()]
rag_scores = [rag.iloc[:, 1].to_list(), rag.iloc[:, 2].to_list(), rag.iloc[:, 3].to_list(), rag.iloc[:, 4].to_list(), rag.iloc[:, 5].to_list()]
bm_scores = [bm.iloc[:, 1].to_list(), bm.iloc[:, 2].to_list(), bm.iloc[:, 3].to_list(), bm.iloc[:, 4].to_list(), bm.iloc[:, 5].to_list()]
gt_scores = [llm_scores.iloc[:, 3].to_list(), llm_scores.iloc[:, 4].to_list(), llm_scores.iloc[:, 5].to_list(), llm_scores.iloc[:, 6].to_list(), llm_scores.iloc[:, 7].to_list()]

query_names = queries#["анемия и мукозит", "лейкоз и антибиотики", "алло-ТГСК и Flu+Bu"]
query_names = [q[:30] for q in queries]

def evaluate_query(y_true, y_pred, k=20):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    
  
    ndcg = ndcg_score([y_true], [y_pred])
   
    corr, _ = spearmanr(y_true, y_pred)
   
    threshold = 0.8
    relevant = y_true >= threshold
    order = np.argsort(y_pred)[::-1]
    relevant_sorted = relevant[order]
    prec_k = np.mean(relevant_sorted[:k]) if len(relevant_sorted) >= k else np.mean(relevant_sorted)
    recall_k = np.sum(relevant_sorted[:k]) / np.sum(relevant) if np.sum(relevant) > 0 else 0.0
    return ndcg, corr, prec_k, recall_k


results_detail = { 'My': {}, 'RAG': {}, 'BM25': {} }
for method_name, scores_list in [('My', my_scores), ('RAG', rag_scores), ('BM25', bm_scores)]:
    for q_idx, q_name in enumerate(query_names):
        ndcg, corr, p_k, r_k = evaluate_query(gt_scores[q_idx], scores_list[q_idx])
        results_detail[method_name][q_name] = (ndcg, corr, p_k, r_k)


print("Метрики по каждому запросу:")
print(f"{'Метод':<6} {'Запрос':<22} {'nDCG':<8} {'Spearman r':<10} {'P@20':<8} {'R@20':<8}")
for method in ['My', 'RAG', 'BM25']:
    for q_name in query_names:
        ndcg, corr, p_k, r_k = results_detail[method][q_name]
        print(f"{method:<6} {q_name:<22} {ndcg:<8.4f} {corr:<10.4f} {p_k:<8.4f} {r_k:<8.4f}")
    print()


print("Средние по всем запросам:")
for method in ['My', 'RAG', 'BM25']:
    ndcg_vals = [results_detail[method][q][0] for q in query_names]
    corr_vals = [results_detail[method][q][1] for q in query_names]
    p_vals = [results_detail[method][q][2] for q in query_names]
    r_vals = [results_detail[method][q][3] for q in query_names]
    print(f"{method}: nDCG={np.mean(ndcg_vals):.4f} ± {np.std(ndcg_vals):.4f}, "
          f"Spearman r={np.mean(corr_vals):.4f} ± {np.std(corr_vals):.4f}, "
          f"P@20={np.mean(p_vals):.4f}, R@20={np.mean(r_vals):.4f}")

Метрики по каждому запросу:
Метод  Запрос                 nDCG     Spearman r P@20     R@20    
My     пациенты с анемией и мукозитом 0.9747   0.7512     0.6500   0.7647  
My     пациенты с лейкозом, принимавш 0.9619   0.5401     0.5500   0.3235  
My     пациенты получившие алло-ТГСК  0.9839   0.5214     0.9000   0.2727  
My     пациенты с острым миелоидным л 0.8849   0.5917     0.0500   0.2000  
My     пациенты с РТПХ кожи принимавш 0.9563   0.6376     0.8000   0.3810  

RAG    пациенты с анемией и мукозитом 0.8692   0.2394     0.2500   0.2941  
RAG    пациенты с лейкозом, принимавш 0.8930   0.1960     0.3500   0.2059  
RAG    пациенты получившие алло-ТГСК  0.9640   0.1789     0.8500   0.2576  
RAG    пациенты с острым миелоидным л 0.8796   0.4981     0.1500   0.6000  
RAG    пациенты с РТПХ кожи принимавш 0.9370   0.3838     0.7000   0.3333  

BM25   пациенты с анемией и мукозитом 0.8439   0.1108     0.1500   0.1765  
BM25   пациенты с лейкозом, принимавш 0.8817   -0.0231    0.1500  